# KazNLP Visualization
Visualize Named Entities and Dependency Trees for Kazakh Text

In [ ]:

from kaznlp.parser.ud_parser import KazakhDependencyParser
from kaznlp.utils.bio_to_bioes import bio_to_bioes
from transformers import AutoTokenizer
from kaznlp.models.ner_bert_bilstm_crf_enhanced import KazBERTNERModel
import torch
import spacy
from spacy.tokens import Doc
from spacy import displacy

# Init models
parser = KazakhDependencyParser()
model_name = "ai-forever/mbert-base-kaznlp"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tag2idx = {"O": 0, "B-LOC": 1, "I-LOC": 2, "E-LOC": 3, "S-LOC": 4, "B-PER": 5, "I-PER": 6, "B-ORG": 7, "I-ORG": 8}
idx2tag = {v: k for k, v in tag2idx.items()}
model = KazBERTNERModel(model_name, tag2idx)
model.load_state_dict(torch.load("kazner_enhanced.pt", map_location="cpu"))
model.eval()


In [ ]:

# Input sentence
text = "Астана — Қазақстанның астанасы."

# NER prediction
tokens = text.split()
inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
with torch.no_grad():
    preds = model(inputs["input_ids"], attention_mask=inputs["attention_mask"])
pred_tags = [idx2tag[tag] for tag in preds[0][:len(tokens)]]

# Parse dependencies
dep_data = parser.parse(text)
for dep in dep_data:
    print(dep)


In [ ]:

# Visualization: Convert to spaCy Doc
nlp = spacy.blank("xx")  # empty pipeline
words = tokens
spaces = [True] * (len(tokens) - 1) + [False]
doc = Doc(nlp.vocab, words=words, spaces=spaces)

# Add ents manually
entities = []
start = 0
for token, tag in zip(words, pred_tags):
    if tag.startswith("B-") or tag.startswith("S-"):
        ent_start = start
    if tag.startswith("E-") or tag.startswith("S-"):
        ent_end = start + len(token)
        ent_type = tag.split("-")[1]
        entities.append({"start": ent_start, "end": ent_end, "label": ent_type})
    start += len(token) + 1

doc.ents = [doc.char_span(e["start"], e["end"], label=e["label"]) for e in entities if doc.char_span(e["start"], e["end"])]

# Render
displacy.render(doc, style="ent", jupyter=True)
